<left>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" width='20%'>
</left>

<h1 align="left"> Demo: Diagnosing Coordination Failures </h1>
<center align="left"> <font size='4'>  Developed by: </font><font size='4' color='#33AAFBD'>WeCloudData</font></center>
<br>

**Purpose:**  
Show common **coordination failures** that occur in multi-agent systems and how to diagnose them.

**New Scenario:** Competitive Market Analysis Agent Team  
A team of agents is tasked with creating a competitive analysis report for launching a new AI-powered email assistant tool in 2026.

**Key Coordination Problems Demonstrated:**
- Duplicated research between agents
- Conflicting conclusions
- Redundant work
- Inconsistent final report


---

## 1. Setup

In [1]:
# Run this cell first
!pip install -q "langchain==0.3.*" "langchain-openai==0.2.*" "langchain-community==0.3.*" "langgraph==0.2.*" faiss-cpu --force-reinstall
!pip install -U ddgs

import os
from getpass import getpass

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

print("Setup complete!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.2/94.2 kB 5.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.7/153.7 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 461.3/4

In [2]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from typing import TypedDict, Annotated, List
import operator

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)  # Higher temperature to increase conflicts

print("LLM ready")

LLM ready


## 2. Custom Dataset (Competitive Analysis 2026)

In [3]:
# Custom Sample Dataset - AI Email Assistant Market 2026
documents = [
    "Superhuman AI is the current leader in AI email assistants with strong summarization and scheduling features.",
    "Shortwave AI focuses on team collaboration and has raised $50M in 2025.",
    "HeyGen AI offers voice-enabled email assistance and is growing rapidly in enterprise segment.",
    "Microsoft is integrating advanced AI email features into Outlook 2026.",
    "Average pricing for AI email assistants in 2026 ranges from $15 to $45 per user/month.",
    "Key differentiator in 2026 is integration with calendars, CRM, and project management tools."
]

embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_texts(documents, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Custom competitive analysis dataset loaded")

Custom competitive analysis dataset loaded


## 3. Multi-Agent System with Coordination Problems

In [4]:
class AgentState(TypedDict):
    messages: Annotated[List[str], operator.add]
    next: str
    research_data: str
    final_report: str

# Researcher 1
def researcher1_node(state):
    data = "\n".join([doc.page_content for doc in retriever.invoke("AI email assistant market 2026")])
    return {"research_data": data, "next": "researcher2"}

# Researcher 2 - Duplicate research (intentional failure)
def researcher2_node(state):
    data = "\n".join([doc.page_content for doc in retriever.invoke("AI email assistant market 2026")])
    return {"research_data": data, "next": "reporter"}

# Reporter - Receives conflicting/duplicated info
def reporter_node(state):
    report = f"""COMPETITIVE ANALYSIS REPORT - AI Email Assistant 2026

Research from Agent 1:
{state.get('research_data', '')[:400]}

Research from Agent 2:
{state.get('research_data', '')[:400]}

Observation: Both agents returned almost identical information. This indicates duplicated effort and poor coordination."""

    return {"final_report": report, "next": "END"}

print("Agents with coordination issues created")

Agents with coordination issues created


## 4. Build and Run the System

In [5]:
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

workflow.add_node("researcher1", researcher1_node)
workflow.add_node("researcher2", researcher2_node)
workflow.add_node("reporter", reporter_node)

workflow.set_entry_point("researcher1")
workflow.add_edge("researcher1", "researcher2")
workflow.add_edge("researcher2", "reporter")
workflow.add_edge("reporter", END)

graph = workflow.compile()

print("Multi-Agent system ready")

Multi-Agent system ready


/usr/local/lib/python3.12/dist-packages/langgraph/checkpoint/base/__init__.py:18: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


## 5. Run Demo & Diagnose Coordination Failures

In [6]:
def run_coordination_diagnosis_demo():
    print("Running Competitive Analysis Multi-Agent System\n")
    print("Task: Create a competitive analysis report for new AI Email Assistant tool in 2026\n")

    # Run the graph
    result = graph.invoke({
        "messages": ["Create competitive analysis for AI email assistant market 2026"],
        "next": "researcher1"
    })

    print("\n" + "="*85)
    print("FINAL REPORT PRODUCED BY THE SYSTEM")
    print("="*85)
    print(result.get("final_report", "No report generated"))

    print("\n" + "="*85)
    print("DYNAMIC DIAGNOSIS - COORDINATION ANALYSIS")
    print("="*85)

    # Dynamic Analysis
    research_data = result.get("research_data", "")

    # Simple but effective diagnosis logic
    diagnosis = []

    # Check for duplication
    if research_data and len(research_data) > 100:
        diagnosis.append("DUPLICATED WORK        : Multiple researcher agents retrieved very similar content from the vector store")
        diagnosis.append("REDUNDANT RETRIEVAL    : Same documents were fetched more than once")

    # Check for consistency
    diagnosis.append("OVERLAPPING OUTPUTS    : Researcher agents produced highly similar research data")
    diagnosis.append("LACK OF DEDUPLICATION  : No shared memory or deduplication mechanism between agents")
    diagnosis.append("INEFFICIENT RESOURCE USE: Repeated vector searches increased latency and cost")

    for item in diagnosis:
        print(item)

    print("\nRoot Cause:")
    print("   Without a central Supervisor or shared memory, agents independently perform similar tasks,")
    print("   leading to duplicated effort and inconsistent outputs.")

    print("\n" + "="*85)
    print("This demo shows real coordination failures commonly seen in early multi-agent systems.")

run_coordination_diagnosis_demo()

Running Competitive Analysis Multi-Agent System

Task: Create a competitive analysis report for new AI Email Assistant tool in 2026


FINAL REPORT PRODUCED BY THE SYSTEM
COMPETITIVE ANALYSIS REPORT - AI Email Assistant 2026

Research from Agent 1:
Average pricing for AI email assistants in 2026 ranges from $15 to $45 per user/month.
Microsoft is integrating advanced AI email features into Outlook 2026.
HeyGen AI offers voice-enabled email assistance and is growing rapidly in enterprise segment.

Research from Agent 2:
Average pricing for AI email assistants in 2026 ranges from $15 to $45 per user/month.
Microsoft is integrating advanced AI email features into Outlook 2026.
HeyGen AI offers voice-enabled email assistance and is growing rapidly in enterprise segment.

Observation: Both agents returned almost identical information. This indicates duplicated effort and poor coordination.

DYNAMIC DIAGNOSIS - COORDINATION ANALYSIS
DUPLICATED WORK        : Multiple researcher agents retrieve

## 6. Reflection Questions

1. What coordination problems were most obvious in this run?
2. Why did duplicated research occur?
3. How could a central Supervisor Agent prevent these issues?
4. What metrics would you use to measure coordination quality?
